# 03 · Fit selectors and freeze the real experiment

**Goal:** choose the teacher and its comparators using source outcomes
alone, then save the decisions used for real deployment.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../../../../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](../../../../../../notebooks/synthetic_training/README.md)
· [HAIC setup and launch commands](../../../../../../slurm/synthetic-training/README.md)

In [1]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

Run: /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/outputs/synthetic-training/pilot-01
Context representation: vjepa; device: cuda


## 1. Give the comparators precisely defined information

Before-only uses the original snapshot. After-only uses the post-probe
snapshot. Before-plus-change receives both the original state and the
transition. All three select training for the same $M_p$ with the same
remaining budget. These are compact summaries, not complete prediction
tensors. Paired target-change features retain signed, absolute, and
RMS displacements. Subtracting two separately pooled snapshots does
not recover all of those paired statistics.

The stronger primary comparison controls for **source learning
progress**. Both selectors receive post-probe target predictions, source
probe loss history, both synthetic diagnostic snapshots, descriptors,
context features, and budget. Only the full teacher also sees target
prediction change. This tests information from unlabeled target video
beyond information from labeled source training. The strict matched
comparator uses the full teacher's selected model family and parameter.
A separately tuned source-progress comparator also tests the strongest
available alternative under the same validation budget.

## 2. Fit on training students; select settings on validation students

Start with nearest-neighbor lookup and small regularized predictors.
Choose settings by the reference error after the selected lesson, not
just by regression fit to the utility numbers. Fit normalization using
training episodes only. A random row split would leak nearly identical
trials across the boundary.

Other controls include fixed/balanced/random lessons, synthetic weakness,
scalar response magnitude, simple context, and equal-budget replay.
A simple selector that uses the response can validate the mechanism.
A neural selector does not have to beat it for the information to help.

In [2]:
fitted = workflow.fit_selectors(cfg)
show_result(fitted)

### selection

{
  "budget": 75,
  "primary_comparator": "domain",
  "best_fixed": "front",
  "methods": {
    "before": {
      "kind": "nearest",
      "parameter": 1.0,
      "validation_error": 0.027025762386308137
    },
    "after": {
      "kind": "nearest",
      "parameter": 1.0,
      "validation_error": 0.027025762386308137
    },
    "response": {
      "kind": "nearest",
      "parameter": 1.0,
      "validation_error": 0.02702220227058258
    },
    "source_progress": {
      "kind": "nearest",
      "parameter": 1.0,
      "validation_error": 0.027025762386308137
    },
    "full": {
      "kind": "nearest",
      "parameter": 1.0,
      "validation_error": 0.02702647999854117
    },
    "magnitude": {
      "kind": "nearest",
      "parameter": 1.0,
      "validation_error": 0.027025762386308137
    },
    "weakness": {
      "kind": "nearest",
      "parameter": 5.0,
      "validation_error": 0.026937256984499655
    },
    "domain": {
      "kind": "nearest",
      "parameter": 3.0,

### validation

,view,kind,parameter,budget,error,regret
27,domain,nearest,3.0,75,0.026781,0.000116
111,simple_context,nearest,3.0,75,0.026842,0.000177
49,image_context,nearest,1.0,75,0.026847,0.000182
113,simple_context,nearest,5.0,75,0.026890,0.000225
29,domain,nearest,5.0,75,0.026898,0.000233
...,...,...,...,...,...,...
119,simple_context,ridge,100.0,75,0.027279,0.000614
83,no_context,ridge,100.0,75,0.027281,0.000616
57,image_context,ridge,10.0,75,0.027292,0.000627
55,image_context,ridge,1.0,75,0.027293,0.000628


## 3. Read the source decision before running real evaluation

The saved selector configuration specifies the primary method,
comparators, feature transformations, and source-selected settings.
No GAVD reference labels or held-architecture lesson outcomes enter this
decision. A prescribed deployment probe on a held student is allowed;
using its lesson outcomes to retune the teacher is not.

A held model's release recipe may be checked for technical compatibility.
Its learning rate must not be optimized against its held lesson outcomes.
If source choices do not improve selected utility over simple controls,
do not describe later real improvements as an established response effect.

Next inspect [07 · Source mechanism report](../../../../../../notebooks/synthetic_training/07_source_mechanism_report.ipynb).
Prepare GAVD only after making the source continuation decision.